HW1 (Foundations - Ch. 2–3):
Normalization & tokenization: select a data set of your choice, conduct a brief ex-
ploratory data analysis, decide an arbitrary task you are supposed to implement, and
apply the text normalization and tokenization as discussed in the classed.
Implement either Edit Distance or Byte-pairing algorithm and apply it to you data set
and analysis the output and comment on it.

In [66]:
import re
from typing import List
import zipfile
import os
import pandas as pd
from itertools import combinations
from tqdm import tqdm

In [2]:
#!/bin/bash
!curl -L -o ~/Downloads/imdb-dataset-of-50k-movie-reviews.zip\
  https://www.kaggle.com/api/v1/datasets/download/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 25.7M  100 25.7M    0     0   513k      0  0:00:51  0:00:51 --:--:--  530k1097k1k


In [8]:
os.makedirs("dataset_imdb", exist_ok=True)


In [9]:
zip_path = os.path.expanduser('~/Downloads/imdb-dataset-of-50k-movie-reviews.zip')
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('dataset_imdb')

In [13]:
imdb_data = pd.read_csv("dataset_imdb/IMDB Dataset.csv")

In [22]:
w = " ".join(imdb_data['review'].fillna('').astype(str))

In [14]:
def tokenize(text: str, pattern: str | None = None) -> List[str]:
    pattern = r"(?:\w+')\w+|(?:[A-Z]\.)+|\w+(?:-\w+)*|[\w+\.]"
    tokens = re.findall(pattern, text)
    return tokens

In [24]:
tokenized_data = tokenize(w)

In [34]:
freq = dict()
vocab = set()

for word in tokenized_data:
    vocab.update(word)
    freq[word] = freq.get(word, 0) + 1

In [49]:
words = {tuple(word): count for word, count in freq.items()}

tmp_dict = dict()
for symb, count in words.items():
    for i in range(len(symb) - 1):
        pair = (symb[i], symb[i + 1])
        tmp_dict[pair] = tmp_dict.get(pair, 0) + count

In [58]:
max_pair = max(tmp_dict, key=tmp_dict.get)
new_words = dict()
for symb, count in words.items():
    i = 0
    new_symb = list()
    while i < len(symb):
        if i < len(symb) - 1 and (symb[i], symb[i + 1]) == max_pair:
            new_symb.append(symb[i] + symb[i + 1])
            i += 2
        else:
            new_symb.append(symb[i])
            i += 1
    new_words[tuple(new_symb)] = count

vocab.add(max_pair[0] + max_pair[1])
words = new_words

In [ ]:
# добавит ьсюла _ или \w
def bpe(data, count_total: int):
    freq = dict()
    vocab = set()

    for word in data:
        vocab.update(word)
        freq[word] = freq.get(word, 0) + 1

    words = {tuple(word): count for word, count in freq.items()}

    for k in tqdm(range(count_total)):
        tmp_dict = dict()
        for symb, count in words.items():
            for i in range(len(symb) - 1):
                pair = (symb[i], symb[i + 1])
                tmp_dict[pair] = tmp_dict.get(pair, 0) + count

        max_pair = max(tmp_dict, key=tmp_dict.get)
        new_words = dict()
        for symb, count in words.items():
            i = 0
            new_symb = list()
            while i < len(symb):
                if i < len(symb) - 1 and (symb[i], symb[i + 1]) == max_pair:
                    new_symb.append(symb[i] + symb[i + 1])
                    i += 2
                else:
                    new_symb.append(symb[i])
                    i += 1
            new_words[tuple(new_symb)] = count

        vocab.add(max_pair[0] + max_pair[1])
        words = new_words

    return vocab

In [98]:
len_unique = len(set(tokenized_data[:12000]))
print(f'Unique size of vocab (without BPE): {len_unique}')

Unique size of vocab (without BPE): 3274


In [96]:
len_stast = len(set(char for word in tokenized_data[:25000] for char in word))
print(f'Vocab. size in the 0 iter of BPE: {len_stast}')

Vocab. size in the 0 iter of BPE: 71


In [97]:
bpe_result = bpe(tokenized_data[:25000], 300)
print(f'Vocab. size in the 300 iter of BPE: {len(bpe_result)}')
print('examples')
len(bpe_result)
for word in bpe_result:
    if len(word) > 2:
        print(word)

100%|██████████| 300/300 [00:01<00:00, 255.55it/s]

Vocab. size in the 300 iter of BPE: 371
examples
con
look
that
most
other
ing
low
ars
ever
into
ack
was
are
ves
ide
bet
fil
have
ment
end
thr
would
who
com
any
ome
ive
much
red
tor
ong
ere
than
act
one
tim
ick
reat
does
ould
our
n't
ame
able
ist
ence
off
and
not
thing
good
with
ber
wor
were
wat
movies
film
very
old
ble
him
more
ause
ice
ind
ful
tain
all
character
ate
movi
where
bec
ver
been
ies
ure
pro
only
his
ple
The
ter
dis
their
ress
ile
even
them
but
though
ted
which
it's
pre
ill
like
could
est
can
has
ght
ari
some
ard
see
its
ell
char
just
now
ite
how
rom
ach
kill
ers
res
urn
movie
this
ain
play
own
for
ally
tory
ust
really
ough
what
per
from
ade
This
ast
ent
ting
man
they
king
ess
her
age
ving
out
ous
about
uch
way
der
charac
ine
time
you
art
ich
ion
scen
get
the
had
ation
ther
ity
ace
ound
irst
watch
ant
story
when
there


In [89]:
tokenized_data[:12000]

['One',
 'of',
 'the',
 'other',
 'reviewers',
 'has',
 'mentioned',
 'that',
 'after',
 'watching',
 'just',
 '1',
 'Oz',
 'episode',
 "you'll",
 'be',
 'hooked',
 '.',
 'They',
 'are',
 'right',
 'as',
 'this',
 'is',
 'exactly',
 'what',
 'happened',
 'with',
 'me',
 '.',
 'br',
 'br',
 'The',
 'first',
 'thing',
 'that',
 'struck',
 'me',
 'about',
 'Oz',
 'was',
 'its',
 'brutality',
 'and',
 'unflinching',
 'scenes',
 'of',
 'violence',
 'which',
 'set',
 'in',
 'right',
 'from',
 'the',
 'word',
 'GO',
 '.',
 'Trust',
 'me',
 'this',
 'is',
 'not',
 'a',
 'show',
 'for',
 'the',
 'faint',
 'hearted',
 'or',
 'timid',
 '.',
 'This',
 'show',
 'pulls',
 'no',
 'punches',
 'with',
 'regards',
 'to',
 'drugs',
 'sex',
 'or',
 'violence',
 '.',
 'Its',
 'is',
 'hardcore',
 'in',
 'the',
 'classic',
 'use',
 'of',
 'the',
 'word',
 '.',
 'br',
 'br',
 'It',
 'is',
 'called',
 'OZ',
 'as',
 'that',
 'is',
 'the',
 'nickname',
 'given',
 'to',
 'the',
 'Oswald',
 'Maximum',
 'Security',